# Niramaya Vision — Colab training notebook
Educational workflow only. Synthetic runs validate plumbing and are not medical models. Save all artifacts and checkpoints to Drive so a Colab disconnect does not erase work.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content
!git clone <YOUR_REPOSITORY_URL> CareCompassAi-Core_Trinity
%cd /content/CareCompassAi-Core_Trinity
!pip install -r ml/requirements-ml.txt
import torch
print('CUDA:', torch.cuda.is_available())
!mkdir -p /content/drive/MyDrive/NiramayaData /content/drive/MyDrive/NiramayaArtifacts

## Add datasets
Option A: upload `kaggle.json` in Colab Files, set its permissions, install Kaggle, and download the dataset slugs. Option B: copy `NiramayaData/ham10000`, `iam`, `rx_bd`, and `lexicon` from Drive. IAM requires registering at the official IAM database site and downloading `words.tgz` plus `ascii.tgz`; if absent, the IAM stage is skipped.

In [ ]:
from pathlib import Path
from google.colab import files
import shutil
source=Path('/content/drive/MyDrive/NiramayaData')
for name in ('ham10000','iam','rx_bd','lexicon'):
    target=Path('ml/data')/name
    if (source/name).exists():
        shutil.copytree(source/name,target,dirs_exist_ok=True)
    else:
        print(f'{source/name} is absent; that dataset stage will report and skip.')
# Optional Kaggle route: upload kaggle.json, then run the documented Kaggle CLI commands after accepting each dataset's terms.

## DermaScan
The custom, EfficientNet and ResNet runs are separate. Use short quick/synthetic mode to verify the pipeline; only evaluate real held-out HAM10000 results for coursework claims. Checkpoints and reports are copied to Drive after each run.

In [ ]:
!python -m ml.derma.train --model custom --quick --synthetic --resume
!cp -r ml/artifacts/derma /content/drive/MyDrive/NiramayaArtifacts/
!cp -r ml/reports/derma /content/drive/MyDrive/NiramayaArtifacts/
# Real data: remove --synthetic and use --data-dir ml/data/ham10000; use --resume after reconnecting.

## RxRead
The IAM reader checks `words.txt` and reports missing/broken images. Synthetic data exercises the CRNN/CTC optimization path but is not an OCR model for use. Do not enable the service for synthetic weights. Train on the supplied IAM/BD word images and verify an independent held-out test set.

In [ ]:
!python -m ml.rx.train --stage bd_only --quick --synthetic --resume
!python -m ml.rx.evaluate
!python -m ml.rx.export
!cp -r ml/artifacts/rx /content/drive/MyDrive/NiramayaArtifacts/
!cp -r ml/reports/rx /content/drive/MyDrive/NiramayaArtifacts/

## Download artifacts
After training and review, zip the artifact/report folders and download them. Unzip into `ml/artifacts/` and `ml/reports/` in the local repository. Model activation must be restricted to genuinely trained checkpoints with independent evaluation; this notebook does not upgrade a bootstrap model's status.

In [ ]:
!cd ml && zip -r /content/niramaya_vision_outputs.zip artifacts reports
from google.colab import files
files.download('/content/niramaya_vision_outputs.zip')